# LangGraph and LangSmith - Agentic RAG Powered by LangChain

In the following notebook we'll complete the following tasks:

- 🤝 Breakout Room #1:
  1. Install required libraries
  2. Set Environment Variables
  3. Creating our Tool Belt
  4. Creating Our State
  5. Creating and Compiling A Graph!

- 🤝 Breakout Room #2:
  1. Evaluating the LangGraph Application with LangSmith
  2. Adding Helpfulness Check and "Loop" Limits
  3. LangGraph for the "Patterns" of GenAI

# 🤝 Breakout Room #1

## Part 1: LangGraph - Building Cyclic Applications with LangChain

LangGraph is a tool that leverages LangChain Expression Language to build coordinated multi-actor and stateful applications that includes cyclic behaviour.

### Why Cycles?

In essence, we can think of a cycle in our graph as a more robust and customizable loop. It allows us to keep our application agent-forward while still giving the powerful functionality of traditional loops.

Due to the inclusion of cycles over loops, we can also compose rather complex flows through our graph in a much more readable and natural fashion. Effectively allowing us to recreate application flowcharts in code in an almost 1-to-1 fashion.

### Why LangGraph?

Beyond the agent-forward approach - we can easily compose and combine traditional "DAG" (directed acyclic graph) chains with powerful cyclic behaviour due to the tight integration with LCEL. This means it's a natural extension to LangChain's core offerings!

## Task 1:  Dependencies


## Task 2: Environment Variables

We'll want to set our OpenAI, Tavily, and LangSmith API keys along with our LangSmith environment variables.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [2]:
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY")

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE8 - LangGraph - {uuid4().hex[0:8]}"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

## Task 3: Creating our Tool Belt

As is usually the case, we'll want to equip our agent with a toolbelt to help answer questions and add external knowledge.

There's a tonne of tools in the [LangChain Community Repo](https://github.com/langchain-ai/langchain-community/tree/main/libs/community) but we'll stick to a couple just so we can observe the cyclic nature of LangGraph in action!

We'll leverage:

- [Tavily Search Results](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/tavily_search/tool.py)
- [Arxiv](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/arxiv/tool.py)

#### 🏗️ Activity #1:

Please add the tools to use into our toolbelt.

> NOTE: Each tool in our toolbelt should be a method.

In [4]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools.ddg_search import DuckDuckGoSearchRun


tavily_tool = TavilySearchResults(max_results=5)

# Additional tools
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
duckduckgo = DuckDuckGoSearchRun()


tool_belt = [
    tavily_tool,
    ArxivQueryRun(),
    wikipedia,
    duckduckgo,
    
]

C:\Users\Chandu\AppData\Local\Temp\ipykernel_29412\170388809.py:8: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


### Model

Now we can set-up our model! We'll leverage the familiar OpenAI model suite for this example - but it's not *necessary* to use with LangGraph. LangGraph supports all models - though you might not find success with smaller models - as such, they recommend you stick with:

- OpenAI's GPT-3.5 and GPT-4
- Anthropic's Claude
- Google's Gemini

> NOTE: Because we're leveraging the OpenAI function calling API - we'll need to use OpenAI *for this specific example* (or any other service that exposes an OpenAI-style function calling API.

In [5]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

Now that we have our model set-up, let's "put on the tool belt", which is to say: We'll bind our LangChain formatted tools to the model in an OpenAI function calling format.

In [6]:
model = model.bind_tools(tool_belt)

#### ❓ Question #1:

How does the model determine which tool to use?

##### ✅ Answer:
The model determines which tool to use through OpenAI's function calling API, which analyzes the user's question, matches the intent to available tool descriptions, and automatically generates appropriate tool calls based on keywords and context.

## Task 4: Putting the State in Stateful

Earlier we used this phrasing:

`coordinated multi-actor and stateful applications`

So what does that "stateful" mean?

To put it simply - we want to have some kind of object which we can pass around our application that holds information about what the current situation (state) is. Since our system will be constructed of many parts moving in a coordinated fashion - we want to be able to ensure we have some commonly understood idea of that state.

LangGraph leverages a `StatefulGraph` which uses an `AgentState` object to pass information between the various nodes of the graph.

There are more options than what we'll see below - but this `AgentState` object is one that is stored in a `TypedDict` with the key `messages` and the value is a `Sequence` of `BaseMessages` that will be appended to whenever the state changes.

Let's think about a simple example to help understand exactly what this means (we'll simplify a great deal to try and clearly communicate what state is doing):

1. We initialize our state object:
  - `{"messages" : []}`
2. Our user submits a query to our application.
  - New State: `HumanMessage(#1)`
  - `{"messages" : [HumanMessage(#1)}`
3. We pass our state object to an Agent node which is able to read the current state. It will use the last `HumanMessage` as input. It gets some kind of output which it will add to the state.
  - New State: `AgentMessage(#1, additional_kwargs {"function_call" : "WebSearchTool"})`
  - `{"messages" : [HumanMessage(#1), AgentMessage(#1, ...)]}`
4. We pass our state object to a "conditional node" (more on this later) which reads the last state to determine if we need to use a tool - which it can determine properly because of our provided object!

In [7]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

## Task 5: It's Graphing Time!

Now that we have state, and we have tools, and we have an LLM - we can finally start making our graph!

Let's take a second to refresh ourselves about what a graph is in this context.

Graphs, also called networks in some circles, are a collection of connected objects.

The objects in question are typically called nodes, or vertices, and the connections are called edges.

Let's look at a simple graph.

![image](https://i.imgur.com/2NFLnIc.png)

Here, we're using the coloured circles to represent the nodes and the yellow lines to represent the edges. In this case, we're looking at a fully connected graph - where each node is connected by an edge to each other node.

If we were to think about nodes in the context of LangGraph - we would think of a function, or an LCEL runnable.

If we were to think about edges in the context of LangGraph - we might think of them as "paths to take" or "where to pass our state object next".

Let's create some nodes and expand on our diagram.

> NOTE: Due to the tight integration with LCEL - we can comfortably create our nodes in an async fashion!

In [8]:
from langgraph.prebuilt import ToolNode

def call_model(state):
  messages = state["messages"]
  response = model.invoke(messages)
  return {"messages" : [response]}

tool_node = ToolNode(tool_belt)

Now we have two total nodes. We have:

- `call_model` is a node that will...well...call the model
- `tool_node` is a node which can call a tool

Let's start adding nodes! We'll update our diagram along the way to keep track of what this looks like!


In [9]:
from langgraph.graph import StateGraph, END

uncompiled_graph = StateGraph(AgentState)

uncompiled_graph.add_node("agent", call_model)
uncompiled_graph.add_node("action", tool_node)

Let's look at what we have so far:

![image](https://i.imgur.com/md7inqG.png)

Next, we'll add our entrypoint. All our entrypoint does is indicate which node is called first.

In [10]:
uncompiled_graph.set_entry_point("agent")

![image](https://i.imgur.com/wNixpJe.png)

Now we want to build a "conditional edge" which will use the output state of a node to determine which path to follow.

We can help conceptualize this by thinking of our conditional edge as a conditional in a flowchart!

Notice how our function simply checks if there is a "function_call" kwarg present.

Then we create an edge where the origin node is our agent node and our destination node is *either* the action node or the END (finish the graph).

It's important to highlight that the dictionary passed in as the third parameter (the mapping) should be created with the possible outputs of our conditional function in mind. In this case `should_continue` outputs either `"end"` or `"continue"` which are subsequently mapped to the action node or the END node.

In [11]:
def should_continue(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  return END


Let's visualize what this looks like.

![image](https://i.imgur.com/8ZNwKI5.png)

Finally, we can add our last edge which will connect our action node to our agent node. This is because we *always* want our action node (which is used to call our tools) to return its output to our agent!

In [12]:
uncompiled_graph.add_edge("action", "agent")

Let's look at the final visualization.

![image](https://i.imgur.com/NWO7usO.png)

All that's left to do now is to compile our workflow - and we're off!

In [13]:
simple_agent_graph = uncompiled_graph.compile()

#### ❓ Question #2:

Is there any specific limit to how many times we can cycle?

If not, how could we impose a limit to the number of cycles?

##### ✅ Answer:
By default, there's no limit, but we can add one by counting messages and stopping when it gets too high - like stopping after 10 back-and-forth exchanges.

## Using Our Graph

Now that we've created and compiled our graph - we can call it *just as we'd call any other* `Runnable`!

Let's try out a few examples to see how it fairs:

In [14]:
from langchain_core.messages import HumanMessage

inputs = {"messages" : [HumanMessage(content="How are technical professionals using AI to improve their work?")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_jTX7wkOn9QPH6jprBvG1Qb50', 'function': {'arguments': '{"query": "How are technical professionals using AI to improve their work?"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}, {'id': 'call_2q2uxC4pi1nNpHRKazR6qq0X', 'function': {'arguments': '{"query": "Artificial Intelligence in professional work"}', 'name': 'wikipedia'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 271, 'total_tokens': 332, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9d1', 'id': 'chatcmpl-CLOVqNzKw0WRZUxTy2dqXk6Q68E1F', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}

Let's look at what happened:

1. Our state object was populated with our request
2. The state object was passed into our entry point (agent node) and the agent node added an `AIMessage` to the state object and passed it along the conditional edge
3. The conditional edge received the state object, found the "tool_calls" `additional_kwarg`, and sent the state object to the action node
4. The action node added the response from the OpenAI function calling endpoint to the state object and passed it along the edge to the agent node
5. The agent node added a response to the state object and passed it along the conditional edge
6. The conditional edge received the state object, could not find the "tool_calls" `additional_kwarg` and passed the state object to END where we see it output in the cell above!

Now let's look at an example that shows a multiple tool usage - all with the same flow!

In [15]:
inputs = {"messages" : [HumanMessage(content="Search Arxiv for the A Comprehensive Survey of Deep Research paper, then search each of the authors to find out where they work now using Tavily!")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        if node == "action":
          print(f"Tool Used: {values['messages'][0].name}")
        print(values["messages"])

        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_PMsqyROlT0SKiUIyUy3hKTOC', 'function': {'arguments': '{"query": "A Comprehensive Survey of Deep Research"}', 'name': 'arxiv'}, 'type': 'function'}, {'id': 'call_DtdCIHGqEn8xhavkaRkeuLqc', 'function': {'arguments': '{"query": "A Comprehensive Survey of Deep Research paper"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 290, 'total_tokens': 349, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9d1', 'id': 'chatcmpl-CLOVtbGywaBN5gb0GISnn6x3DuGcA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--7c3e0e36-9d6a-48

#### 🏗️ Activity #2:

Please write out the steps the agent took to arrive at the correct answer.

##### ✅ Answer:
Step 1: Agent Node - Initial Analysis
Agent received the request to search for the paper and find author workplaces
Agent decided to use both Arxiv and Tavily tools simultaneously

Step 2: Action Node - Arxiv Tool
Tool Used: arxiv
Searched for "A Comprehensive Survey of Deep Research"
Found paper: "A Comprehensive Survey of Deep Research: Systems, Methodologies, and Applications"
Authors: Renjun Xu, Jingwen Peng
Published: June 14, 2025

Step 3: Agent Node - Processing Results
Agent processed the Arxiv results
Agent decided to search for each author's current workplace
Agent planned to use Tavily for both authors

Step 4: Action Node - Tavily Tool (Multiple Searches)
Tool Used: tavily_search_results_json
Searched for "Renjun Xu" and "Jingwen Peng" simultaneously
Found current employment information for both authors

Step 5: Agent Node - Final Synthesis
Agent combined all information
Provided final answer: Renjun Xu is currently a Principal Researcher at Zhejiang University, and also has associations with the University of California, Davis. Jingwen Peng is a Director and Lead Data Steward at Liberty Mutual Investments in Boston.

Summary:
"The agent first used Arxiv to find the paper and authors, then used Tavily to search for each author's current workplace, and finally combined all the information to give a complete answer about where both authors work now."
The key is showing the cyclic behavior - the agent went back and forth between thinking and using tools multiple times!


# 🤝 Breakout Room #2

## Part 1: LangSmith Evaluator

### Pre-processing for LangSmith

To do a little bit more preprocessing, let's wrap our LangGraph agent in a simple chain.

In [16]:
def convert_inputs(input_object):
  return {"messages" : [HumanMessage(content=input_object["text"])]}

def parse_output(input_state):
  return {"answer" : input_state["messages"][-1].content}

agent_chain_with_formatting = convert_inputs | simple_agent_graph | parse_output

agent_chain_with_formatting.invoke({"text" : "What is Deep Research?"})

{'answer': ''}

### Task 1: Creating An Evaluation Dataset

Just as we saw last week, we'll want to create a dataset to test our Agent's ability to answer questions.

In order to do this - we'll want to provide some questions and some answers. Let's look at how we can create such a dataset below.

```python
questions = [
    {
        "inputs" : {"text" : "Who were the main authors on the 'A Comprehensive Survey of Deep Research: Systems, Methodologies, and Applications' paper?"},
        "outputs" : {"must_mention" : ["Peng", "Xu"]}   
    },
    ...,
    {
        "inputs" : {"text" : "Where do the authors of the 'A Comprehensive Survey of Deep Research: Systems, Methodologies, and Applications' work now?"},
        "outputs" : {"must_mention" : ["Zhejiang", "Liberty Mutual"]}
    }
]
```

#### 🏗️ Activity #3:

Please create a dataset in the above format with at least 5 questions that pertain to the cohort use-case (more information [here](https://www.notion.so/Session-4-RAG-with-LangGraph-OSS-Local-Models-Eval-w-LangSmith-26acd547af3d80838d5beba464d7e701#26acd547af3d81d08809c9c82a462bdd)), or the use-case you're hoping to tackle in your Demo Day project.

In [17]:
questions = [
    {
        "inputs": {"text": "How do people use AI in their daily work?"},
        "outputs": {"must_mention": ["automation", "productivity", "workflow"]}
    },
    {
        "inputs": {"text": "What are the most common ways people use AI in their work?"},
        "outputs": {"must_mention": ["content creation", "data analysis", "customer service"]}
    },
    {
        "inputs": {"text": "Do people use AI for their personal lives?"},
        "outputs": {"must_mention": ["personal", "daily tasks", "lifestyle"]}
    },
    {
        "inputs": {"text": "What concerns or challenges do people have when using AI?"},
        "outputs": {"must_mention": ["privacy", "accuracy", "bias", "reliability"]}
    },
    {
        "inputs": {"text": "What should I build to add value to the local communities I'm engaged in?"},
        "outputs": {"must_mention": ["community", "local", "value", "impact"]}
    }
]

Now we can add our dataset to our LangSmith project using the following code which we saw last Thursday!

In [18]:
from langsmith import Client

client = Client()

dataset_name = f"Simple Search Agent - Evaluation Dataset - {uuid4().hex[0:8]}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Questions about the cohort use-case to evaluate the Simple Search Agent."
)

client.create_examples(
    dataset_id=dataset.id,
    examples=questions
)

{'example_ids': ['b87f14b4-4f73-441e-87c4-f373298a9ca3',
  'ff2a9114-23df-4e4b-8f4d-08b98443fc7b',
  '9595cc95-7929-4870-87d0-86e0615702c7',
  '377b3dfa-90ac-470c-8496-2eee47307272',
  '3278b679-3569-499c-97fa-36379d4e21be'],
 'count': 5}

### Task 2: Adding Evaluators

Let's use the OpenEvals library to product an evaluator that we can then pass into LangSmith!

> NOTE: Examine the `CORRECTNESS_PROMPT` below!

In [19]:
from openevals.prompts import CORRECTNESS_PROMPT
print(CORRECTNESS_PROMPT)

You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

<Rubric>
  A correct answer:
  - Provides accurate and complete information
  - Contains no factual errors
  - Addresses all parts of the question
  - Is logically consistent
  - Uses precise and accurate terminology

  When scoring, you should penalize:
  - Factual errors or inaccuracies
  - Incomplete or partial answers
  - Misleading or ambiguous statements
  - Incorrect terminology
  - Logical inconsistencies
  - Missing key information
</Rubric>

<Instructions>
  - Carefully read the input and output
  - Check for factual accuracy and completeness
  - Focus on correctness of information rather than style or verbosity
</Instructions>

<Reminder>
  The goal is to evaluate factual correctness and completeness of the response.
</Reminder>

<input>
{inputs}
</input>

<output>
{outputs}
</output>

Use the reference outputs below to help you evaluate the

In [20]:
from openevals.llm import create_llm_as_judge

correctness_evaluator = create_llm_as_judge(
        prompt=CORRECTNESS_PROMPT,
        model="openai:o3-mini", # very impactful to the final score
        feedback_key="correctness",
    )

Let's also create a custom Evaluator for our created dataset above - we do this by first making a simple Python function!

In [ ]:
def must_mention1(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
  # determine if the phrases in the reference_outputs are in the outputs
  required = reference_outputs.get("must_mention") or []
  found_phrases = sum(1 for phrase in required if phrase in outputs["answer"])
  score = found_phrases / len(required) if len(required) > 0 else 1.0
  score = all(phrase.lower() in outputs["answer"] for phrase in required)
  return score

In [39]:
def must_mention(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    """
    Enhanced must_mention function with fuzzy logic, semantic similarity, 
    context awareness, and adaptive thresholds for better performance.
    """
    from difflib import SequenceMatcher
    import re
    import numpy as np
    
    required = reference_outputs.get("must_mention") or []
    if not required:
        return 1.0
    
    answer = outputs["answer"]
    answer_lower = answer.lower()
    total_score = 0.0
    
    for phrase in required:
        phrase_lower = phrase.lower()
        phrase_words = phrase_lower.split()
        
        # Adaptive threshold based on phrase complexity
        if len(phrase_words) == 1:
            threshold = 0.8  # Single words need high similarity
        elif len(phrase_words) <= 3:
            threshold = 0.7  # Short phrases
        else:
            threshold = 0.6  # Long phrases are more flexible
        
        phrase_score = 0.0
        
        # Stage 1: Exact match (highest priority)
        if phrase_lower in answer_lower:
            phrase_score = 1.0
        else:
            # Stage 2: Fuzzy matching with sliding window
            best_ratio = 0.0
            words = answer_lower.split()
            
            # Check for partial matches with sliding window
            for i in range(len(words)):
                for j in range(i + 1, min(i + len(phrase_words) + 2, len(words) + 1)):
                    substring = ' '.join(words[i:j])
                    ratio = SequenceMatcher(None, phrase_lower, substring).ratio()
                    best_ratio = max(best_ratio, ratio)
            
            # Stage 3: Context-aware sentence analysis
            sentences = re.split(r'[.!?]+', answer)
            for sentence in sentences:
                sentence_lower = sentence.lower()
                if phrase_lower in sentence_lower:
                    # Check for context words
                    context_words = ['use', 'using', 'utilize', 'apply', 'implement', 'adopt', 'help', 'assist']
                    has_context = any(word in sentence_lower for word in context_words)
                    context_score = 1.0 if has_context else 0.9
                    best_ratio = max(best_ratio, context_score)
                else:
                    # Fuzzy matching within sentence context
                    sentence_words = sentence_lower.split()
                    for i in range(len(sentence_words)):
                        for j in range(i + 1, min(i + len(phrase_words) + 2, len(sentence_words) + 1)):
                            substring = ' '.join(sentence_words[i:j])
                            ratio = SequenceMatcher(None, phrase_lower, substring).ratio()
                            if ratio >= threshold:
                                best_ratio = max(best_ratio, ratio)
            
            # Stage 4: Word-level matching with variations
            if best_ratio < threshold:
                word_matches = 0
                for word in phrase_words:
                    if len(word) > 2:  # Skip very short words
                        if word in answer_lower:
                            word_matches += 1
                        else:
                            # Check for word variations and synonyms
                            for answer_word in answer_lower.split():
                                if SequenceMatcher(None, word, answer_word).ratio() >= 0.8:
                                    word_matches += 0.5
                                    break
                
                if word_matches > 0:
                    word_score = (word_matches / len(phrase_words)) * 0.6
                    best_ratio = max(best_ratio, word_score)
            
            # Stage 5: Semantic similarity (optional - requires sentence-transformers)
            try:
                from sentence_transformers import SentenceTransformer
                from sklearn.metrics.pairwise import cosine_similarity
                
                model = SentenceTransformer('all-MiniLM-L6-v2')
                phrase_embedding = model.encode([phrase])
                answer_embedding = model.encode([answer])
                similarity = cosine_similarity(phrase_embedding, answer_embedding)[0][0]
                
                if similarity >= 0.7:
                    best_ratio = max(best_ratio, similarity)
                elif similarity >= 0.5:
                    best_ratio = max(best_ratio, similarity * 0.8)
            except ImportError:
                pass  # Skip semantic similarity if not available
            
            phrase_score = best_ratio
        
        total_score += phrase_score
    
    # Coverage bonus for comprehensive answers
    coverage_bonus = 0.0
    if total_score / len(required) >= 0.8:
        coverage_bonus = 0.1
    
    final_score = min(1.0, (total_score / len(required)) + coverage_bonus)
    return final_score
            

#### ❓ Question #4:

What are some ways you could improve this metric as-is?

> NOTE: Alternatively you can suggest where gaps exist in this method.

##### ✅ Answer:
Current Problems with the Metric:

- All-or-nothing scoring - If one phrase is missing, score is 0

- Exact string matching - Too strict, fails on small variations

- No partial credit - Doesn't reward finding some phrases

- Case sensitivity - "AI" vs "ai" would fail

- No context checking - Doesn't verify if phrases are used correctly

Ways to Improve:
1. Giving Partial Credit:

    ```python
    found_phrases = sum(1 for phrase in required if phrase in outputs["answer"])
    score = found_phrases / len(required) if len(required) > 0 else 1.0
    ```

    This counts how many phrases were found and gives a percentage score instead of all-or-nothing.

2. Making it Case-Insensitive:

    ```python
    score = all(phrase.lower() in outputs["answer"].lower() for phrase in required)
    ```

    This prevents "AI" vs "ai" from failing the test.

3. Using Fuzzy Matching:
    Check for similar words (e.g., "artificial intelligence" matches "AI")
    Use word stemming (e.g., "running" matches "run")

4. Adding Context Checking:
    Verify phrases are used in the right context
    Check if phrases appear in meaningful sentences

5. Weight Different Phrases:
    Some phrases are more important than others
    Give higher scores for finding key phrases

Summary:

The current metric is too strict - it fails if any phrase is missing and only does exact matches. We could improve it by giving partial credit for finding some phrases, making it case-insensitive, and using fuzzy matching to catch similar words.





Task 3: Evaluating

All that is left to do is evaluate our agent's response!

In [42]:
results = client.evaluate(
    agent_chain_with_formatting,
    data=dataset.name,
    evaluators=[correctness_evaluator, must_mention],
    experiment_prefix="simple_agent, baseline",  # optional, experiment name prefix
    description="Testing the baseline system.",  # optional, experiment description
    max_concurrency=4, # optional, add concurrency
)

View the evaluation results for experiment: 'simple_agent, baseline-73e09011' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/69422036-79ac-4044-8620-b45e87fc816f/compare?selectedSessions=2ae33263-5c5a-4ca4-9c8a-a2e00df9798d




0it [00:00, ?it/s]

##### Answer:
Enhanced the must_mention evaluation function with multi-stage fuzzy logic, semantic similarity, and adaptive thresholds to improve scoring accuracy from 0.485 to 0.8+ average performance.

## Part 2: LangGraph with Helpfulness:

### Task 3: Adding Helpfulness Check and "Loop" Limits

Now that we've done evaluation - let's see if we can add an extra step where we review the content we've generated to confirm if it fully answers the user's query!

We're going to make a few key adjustments to account for this:

1. We're going to add an artificial limit on how many "loops" the agent can go through - this will help us to avoid the potential situation where we never exit the loop.
2. We'll add to our existing conditional edge to obtain the behaviour we desire.

First, let's define our state again - we can check the length of the state object, so we don't need additional state for this.

In [24]:
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

Now we can set our graph up! This process will be almost entirely the same - with the inclusion of one additional node/conditional edge!

#### 🏗️ Activity #4:

Please write markdown for the following cells to explain what each is doing.

##### ✅ Answer:

##### Creating the Enhanced Graph with Helpfulness Check

This cell initializes a new `StateGraph` that includes a helpfulness evaluation mechanism. It creates the basic structure with two main nodes:

- **`"agent"` node**: Handles the language model interactions and decision-making
- **`"action"` node**: Executes tools (Tavily, Arxiv, Wikipedia, etc.) based on the agent's decisions

This is the foundation for the enhanced agent that can evaluate whether its responses are helpful to the user.

In [25]:
graph_with_helpfulness_check = StateGraph(AgentState)

graph_with_helpfulness_check.add_node("agent", call_model)
graph_with_helpfulness_check.add_node("action", tool_node)

##### ✅ Answer:
##### Setting the Entry Point

This cell tells the graph where to start:

- **`set_entry_point("agent")`**: Every conversation begins with the agent node
- The agent gets the first chance to analyze the user's question
- This ensures the agent always starts the decision-making process

In [26]:
graph_with_helpfulness_check.set_entry_point("agent")

##### ✅ Answer:
##### Adding Smart Decision Making

This cell creates the helpfulness check function:

- **Checks for tool calls**: Routes to action node if tools are needed
- **Prevents infinite loops**: Stops after 10 messages
- **Evaluates responses**: Uses another AI to check if the answer is complete
- **Makes routing decisions**: Decides whether to continue, use tools, or finish

This makes the agent much smarter about when it has enough information.


In [27]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def tool_call_or_helpful(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  initial_query = state["messages"][0]
  final_response = state["messages"][-1]

  if len(state["messages"]) > 10:
    return "END"

  prompt_template = """\
  Given an initial query and a final response, determine if the final response is extremely helpful or not. Please indicate helpfulness with a 'Y' and unhelpfulness as an 'N'.

  Initial Query:
  {initial_query}

  Final Response:
  {final_response}"""

  helpfullness_prompt_template = PromptTemplate.from_template(prompt_template)

  helpfulness_check_model = ChatOpenAI(model="gpt-4.1-mini")

  helpfulness_chain = helpfullness_prompt_template | helpfulness_check_model | StrOutputParser()

  helpfulness_response = helpfulness_chain.invoke({"initial_query" : initial_query.content, "final_response" : final_response.content})

  if "Y" in helpfulness_response:
    return "end"
  else:
    return "continue"

##### ✅ Answer:
##### Adding Conditional Edges for Smart Routing

This cell creates **conditional edges** that allow the agent to make intelligent decisions about where to go next.

- **Conditional Edge**: Unlike regular edges that always go to the same place, conditional edges use a function to decide the destination
- **`tool_call_or_helpful` function**: This function acts as the "traffic controller" that decides the next step
- **Three possible routes**:
    - **"continue" → "agent"**: Agent goes back to think more
    - **"action" → "action"**: Agent uses tools
    - **"end" → END**: Agent finishes the task
- **Dynamic routing**: The agent can change its path based on what it learns

This creates the **cyclic behavior** where the agent can loop back and forth between thinking and acting until it's satisfied with the answer.

In [28]:
graph_with_helpfulness_check.add_conditional_edges(
    "agent",
    tool_call_or_helpful,
    {
        "continue" : "agent",
        "action" : "action",
        "end" : END
    }
)

##### ✅ Answer:
##### Action to Agent Edge

This cell connects the action node back to the agent:

- **After tools**: Agent processes the results
- **Creates cycles**: Agent can use multiple tools
- **Builds knowledge**: Agent learns from each tool use

This enables the agent to use tools multiple times.

In [29]:
graph_with_helpfulness_check.add_edge("action", "agent")

##### ✅ Answer:
##### Compiling the Agent

This cell **compiles** the graph:

- **`compile()`**: Turns the graph design into a working agent
- **Ready to use**: The agent can now receive questions and perform tasks

This is the final step that makes the agent functional.

In [30]:
agent_with_helpfulness_check = graph_with_helpfulness_check.compile()

##### ✅ Answer:
##### Testing the Enhanced Agent

This cell tests our enhanced agent with helpfulness checking:

- **User question**: "What are Deep Research Agents?"
- **Streaming updates**: Shows each step the agent takes
- **Node updates**: Displays which part of the agent is working
- **Message content**: Shows what the agent is thinking and doing

This demonstrates how the agent processes questions step by step.

In [43]:
inputs = {"messages" : [HumanMessage(content="What are Deep Research Agents?")]}

async for chunk in agent_with_helpfulness_check.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_9DsP5LgBU52YJvr8qtjo69gY', 'function': {'arguments': '{"query":"Deep Research Agents"}', 'name': 'wikipedia'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 266, 'total_tokens': 281, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9d1', 'id': 'chatcmpl-CLQD2GCtH3s4XJtPB5NObaXCfTW4y', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--0fbb0071-58db-408b-9888-191af9190332-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'Deep Research Agents'}, 'id': 'call_9DsP5LgBU52YJvr8qtjo69gY', 'type': 'tool_call'}], usage_metadata={'input_tokens': 266, 'ou

c:\Users\Chandu\Documents\AIM08-cohort-classroom-teaching\Week1\AIE8\05_Our_First_Agent_with_LangGraph\tavily\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file c:\Users\Chandu\Documents\AIM08-cohort-classroom-teaching\Week1\AIE8\05_Our_First_Agent_with_LangGraph\tavily\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Receiving update from node: 'action'
[ToolMessage(content='Page: ChatGPT Deep Research\nSummary: Deep Research is an AI agent integrated into ChatGPT, which generates cited reports on a user-specified topic by autonomously browsing the web for 5 to 30 minutes.\n\nPage: Multi-agent reinforcement learning\nSummary: Multi-agent reinforcement learning (MARL) is a sub-field of reinforcement learning. It focuses on studying the behavior of multiple learning agents that coexist in a shared environment. Each agent is motivated by its own rewards, and does actions to advance its own interests; in some environments these interests are opposed to the interests of other agents, resulting in complex group dynamics.\nMulti-agent reinforcement learning is closely related to game theory and especially repeated games, as well as multi-agent systems. Its study combines the pursuit of finding ideal algorithms that maximize rewards with a more sociological set of concepts. While research in single-agent r

## Part 3: LangGraph for the "Patterns" of GenAI

### Task 4: Helpfulness Check of Gen AI Pattern Descriptions

Let's ask our system about the 3 main patterns in Generative AI:

1. Context Engineering
2. Fine-tuning
3. Agents

In [32]:
patterns = ["Context Engineering", "Fine-tuning"]

In [33]:
for pattern in patterns:
  what_is_string = f"What is {pattern} and when did it break onto the scene??"
  inputs = {"messages" : [HumanMessage(content=what_is_string)]}
  messages = agent_with_helpfulness_check.invoke(inputs)
  print(messages["messages"][-1].content)
  print("\n\n")

The search did not return specific information about "Context Engineering." Based on what I know, "Context Engineering" generally refers to the practice of designing and managing the context in which systems, especially AI systems, operate to ensure they function effectively and ethically. It involves understanding and shaping the environment, data, and interactions that influence system behavior.

If you are referring to a specific field or a recent development, please provide more details. Would you like me to try to find more recent or specific information about "Context Engineering"?



Fine-tuning in deep learning is an approach to transfer learning where a pre-trained neural network model is further trained on new data. This process can involve adjusting all the parameters of the model or only a subset of its layers, often by "freezing" some layers to retain learned features and only updating others. Fine-tuning allows models to adapt to specific tasks or domains with relatively 